# Strands Agents com AgentCore Memory (Memória de Curto Prazo) - Usando MemoryManager


## Introdução

Este tutorial demonstra como construir um **agente pessoal** usando Strands agents com **memória de curto prazo** do AgentCore usando **MemoryManager** e **MemorySessionManager**. O agente lembra conversas recentes na sessão usando `get_last_k_turns` e pode continuar conversas de forma transparente quando o usuário retorna.

**NOTA: Esta é a versão do exemplo de Memória de Curto Prazo usando MemoryManager & MemorySessionManager.**


### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Curto Prazo Conversacional                                                       |
| Tipo de agente      | Agente Pessoal                                                                   |
| Framework Agêntico  | Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | Memória de Curto Prazo AgentCore, MemoryManager, MemorySessionManager, hooks AgentInitializedEvent e MessageAddedEvent |
| Complexidade do exemplo | Iniciante                                                                    |

Você aprenderá a:
- Usar o MemoryManager para gerenciar ciclo de vida de memória
- Usar o MemorySessionManager para gerenciar sessões
- Usar memória de curto prazo para continuidade de conversas
- Recuperar as últimas K rodadas de conversa
- Ferramenta de busca web para informações em tempo real
- Inicializar agentes com histórico de conversas

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Pré-requisitos

- Python 3.10+
- Credenciais AWS com permissões do AgentCore Memory
- ARN da role do AgentCore Memory
- Acesso aos modelos do Amazon Bedrock

Vamos começar configurando nosso ambiente!

## Passo 1: Configuração e Imports

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from botocore.exceptions import ClientError

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("personal-agent")

In [ ]:
# Import required modules for Strands Agent
import os
from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# Import memory management modules
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# Define message role constants
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

# Configuration
REGION = os.getenv('AWS_REGION', 'us-east-1') # AWS region for the agent
ACTOR_ID = "user_123" # It can be any unique identifier (AgentID, User ID, etc.)
SESSION_ID = "personal_session_001" # Unique session identifier

# Import boto3 for IAM role creation
import boto3
import json as json_module

## Passo 2: Ferramenta de Busca Web

Primeiro, vamos criar uma ferramenta simples de busca web para o agente.

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("✅ Web search tool ready")

## Passo 3: Criar Recurso de Memória usando MemoryManager

Para memória de curto prazo, criamos um recurso de memória sem nenhuma estratégia usando o MemoryManager. Isso armazena rodadas de conversa brutas que podem ser recuperadas com `get_last_k_turns`.

In [ ]:
# Initialize Memory Manager 
memory_manager = MemoryManager(region_name=REGION)
memory_name = "PersonalAgentMemoryManager"

logger.info(f"✅ MemoryManager initialized for region: {REGION}")
logger.info(f"Memory manager type: {type(memory_manager)}")

# Create memory resource using MemoryManager
logger.info(f"Creating memory '{memory_name}' for short-term conversational storage...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=[],  # No strategies for short-term memory
        description="Short-term memory for personal agent",
        event_expiry_days=7,  # Retention period for short-term memory
        memory_execution_role_arn=None,  # Optional for short-term memory
    )
    memory_id = memory.id
    logger.info(f"✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")
    
except Exception as e:
    # Handle any errors during memory creation with enhanced error reporting
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")
    
    # Re-raise the original exception
    raise

## Passo 4: Inicializar o Gerenciador de Sessão

Esta seção apresenta o MemorySessionManager que lida com gerenciamento de sessão, incluindo `get_last_k_turns`, adição de rodadas e mais. Ele simplifica as interações de memória para nosso agente.

In [ ]:
# Initialize the session memory manager
session_manager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# Create a memory session for the specific actor/session combination
user_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID, 
    session_id=SESSION_ID
)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"✅ Memory session created for actor: {ACTOR_ID}, session: {SESSION_ID}")
logger.info(f"Session manager type: {type(session_manager)}")
logger.info(f"Memory session type: {type(user_session)}")

## Passo 5: Provedor de Hook de Memória

Este passo define nossa classe customizada `MemoryHookProvider` que automatiza operações de memória. Hooks são funções especiais que executam em pontos específicos do ciclo de vida de execução de um agente. O hook de memória que estamos criando serve duas funções principais:
1. **Carregar conversa recente**: Usamos o hook `AgentInitializedEvent` que carregará automaticamente o histórico de conversa recente quando o agente é inicializado.
2. **Armazenar a última mensagem**: Armazena nova mensagem conversacional.

Isso cria uma experiência de memória transparente sem gerenciamento manual.

In [ ]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_session: MemorySession):  # Accept MemorySession instead
        self.memory_session = memory_session
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts using MemorySession"""
        try:
            # Use the pre-configured memory session (no need for actor_id/session_id)
            recent_turns = self.memory_session.get_last_k_turns(k=5)
            
            if recent_turns:
                # Format conversation history for context
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        # Handle both EventMessage objects and dict formats
                        if hasattr(message, 'role') and hasattr(message, 'content'):
                            role = message['role']
                            content = message['content']
                        else:
                            role = message.get('role', 'unknown')
                            content = message.get('content', {}).get('text', '')
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                # Add context to agent's system prompt
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns using MemorySession")
                
        except Exception as e:
            logger.error(f"Memory load error: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory using MemorySession"""
        messages = event.agent.messages
        try:
            if messages and len(messages) > 0 and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = MessageRole.USER if messages[-1]["role"] == "user" else MessageRole.ASSISTANT
                
                # Use memory session instance (no need to pass actor_id/session_id)
                result = self.memory_session.add_turns(
                    messages=[ConversationalMessage(message_text, message_role)]
                )
                
                event_id = result['eventId']
                logger.info(f"✅ Stored message with Event ID: {event_id}, Role: {message_role.value}")
                
        except Exception as e:
            logger.error(f"Memory save error: {e}")
            import traceback
            logger.error(f"Full traceback: {traceback.format_exc()}")
    
    def register_hooks(self, registry: HookRegistry):
        # Register memory hooks
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        logger.info("✅ Memory hooks registered with MemorySession")


## Passo 6: Criar Agente Pessoal com Busca Web

Este agente usa o MemoryHookProvider com MemorySessionManager para lidar automaticamente com armazenamento e recuperação de memória.

In [ ]:
def create_personal_agent():
    """Create personal agent with memory and web search using MemorySession"""
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        system_prompt=f"""You are a helpful personal assistant with web search capabilities.
        
        You can help with:
        - General questions and information lookup
        - Web searches for current information
        - Personal task management
        
        When you need current information, use the websearch function.
        Today's date: {datetime.today().strftime('%Y-%m-%d')}
        Be friendly and professional.""",
        hooks=[MemoryHookProvider(user_session)], 
        tools=[websearch],
    )
    return agent

# Create agent
agent = create_personal_agent()
logger.info("✅ Personal agent created with MemorySession and web search")

#### Parabéns! Seu agente está pronto com o MemoryManager & MemorySessionManager! :)
## Vamos testar o Agente

In [ ]:
# Test conversation with memory
print("=== First Conversation ===")
print(f"User: My name is Alex and I'm interested in learning about AI.")
print(f"Agent: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

In [ ]:
print(f"User: Can you search for the latest AI trends in 2025?")
print(f"Agent: ", end="")
agent("Can you search for the latest AI trends in 2025?")

In [ ]:
print(f"User: I'm particularly interested in machine learning applications.")
print(f"Agent: ", end="")
agent("I'm particularly interested in machine learning applications.")

## Testar Continuidade de Memória com MemorySessionManager

Para testar se nosso sistema de memória está funcionando corretamente, vamos criar uma nova instância do agente e ver se ele consegue acessar as informações previamente armazenadas:

In [ ]:
# Create new agent instance (simulates user returning)
print("=== User Returns - New Session ===")
new_agent = create_personal_agent()

# Test memory continuity
print(f"User: What was my name again?")
print(f"Agent: ", end="")
new_agent("What was my name again?")

print(f"User: Can you search for more information about machine learning?")
print(f"Agent: ", end="")
new_agent("Can you search for more information about machine learning?")

## Visualizar Memória Armazenada usando MemorySession

In [ ]:
# Check what's stored in memory using MemorySession
print("=== Memory Contents ===")
recent_turns = user_session.get_last_k_turns(k=3) 

for i, turn in enumerate(recent_turns, 1):
    print(f"Turn {i}:")
    for message in turn:
        role = message['role']
        content = message['content']['text'][:100] + "..." if len(message['content']['text']) > 100 else message['content']['text']
        print(f"  {role}: {content}")
    print()

## Resumo

Este tutorial mostrou como construir um agente pessoal usando tanto MemoryManager quanto MemorySessionManager. Você aprendeu:

- Usar o MemoryManager para criar e gerenciar recursos de memória
- Usar o MemorySessionManager para operações de sessão
- Criar recursos de memória sem estratégias
- Usar `get_last_k_turns` para histórico de conversas
- Adicionar capacidades de busca web aos agentes
- Implementar hooks de memória para carregamento de contexto

**Próximos Passos:**
- Adicionar ferramentas mais sofisticadas
- Implementar estratégias de memória de longo prazo
- Aprimorar capacidades de busca com múltiplas fontes

## Limpeza (Opcional)

In [ ]:
# Clean up all resources created during notebook execution
print("=== Limpeza de Recursos ===")

# Step 1: Delete all events in the session (short-term memory data)
try:
    events = user_session.list_events(max_results=100)
    if events:
        logger.info(f"🗑️ Deleting {len(events)} events from session...")
        for event in events:
            event_id = event.get('eventId', event.get('id', None))
            if event_id:
                user_session.delete_event(event_id)
        logger.info(f"✅ All events deleted from session: {SESSION_ID}")
    else:
        logger.info("ℹ️ No events found in session to delete.")
except Exception as e:
    logger.warning(f"⚠️ Could not delete events (may not exist): {e}")

# Step 2: Delete the Memory resource (and wait for completion)
# This removes the memory resource and all associated data (actors, sessions, events)
try:
    logger.info(f"🗑️ Deleting memory resource: {memory_id}")
    memory_manager.delete_memory_and_wait(memory_id)
    logger.info(f"✅ Memory resource fully deleted: {memory_id}")
except Exception as e:
    logger.error(f"❌ Failed to delete memory: {e}")

print("=== Limpeza concluída ===")